# 00 — Differential Context in Noisy Sensing

This notebook builds the first toy model for `differential-context-sensing`.

The core idea is simple: two sensors can look unusably noisy on their own while their difference preserves the physical signal, provided the dominant disturbance is shared.

Paper context: arXiv:2504.09158 studies AION atom interferometry and demonstrates common-mode laser phase noise rejection while maintaining Standard Quantum Limit behavior.

## 1. Setup

We use NumPy for simulation, pandas for the summary table, and matplotlib for figures. Paths are resolved relative to the repository root so the notebook works locally or in Colab after cloning the repo.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from differential_context_sensing.paths import DATA, FIGURES, ensure_dirs

ensure_dirs()

print(f"Repository root: {ROOT}")
print(f"Figures: {FIGURES}")
print(f"Data: {DATA}")

## 2. Model

Sensor A and Sensor B each measure a physical phase.

Both sensors receive:

- a true differential signal,
- shared common-mode noise,
- independent local noise.

The observed phases are:

\[
\phi_A = +\frac{s}{2} + n_{common} + n_A
\]

\[
\phi_B = -\frac{s}{2} + n_{common} + n_B
\]

The differential phase is:

\[
\phi_{diff} = \phi_A - \phi_B = s + n_A - n_B
\]

The shared noise cancels.

In [ ]:
rng = np.random.default_rng(42)

n = 1000
t = np.linspace(0, 10, n)

signal = 0.3 * np.sin(2 * np.pi * 0.4 * t)
common_noise = 2.0 * rng.normal(size=n)
local_a = 0.15 * rng.normal(size=n)
local_b = 0.15 * rng.normal(size=n)

phi_a = 0.5 * signal + common_noise + local_a
phi_b = -0.5 * signal + common_noise + local_b
phi_diff = phi_a - phi_b

print("Simulated two sensors with dominant shared noise.")

## 3. Individual sensors

Each sensor is dominated by the same large common-mode disturbance. Looking at one sensor alone, the physical signal is visually buried.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, phi_a, label="Sensor A")
plt.plot(t, phi_b, label="Sensor B", alpha=0.8)
plt.xlabel("time")
plt.ylabel("phase")
plt.title("Individual sensors are dominated by shared noise")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "00_individual_sensors.png", dpi=200)
plt.show()

## 4. Differential recovery

Subtracting the two sensors removes the shared disturbance. The differential channel tracks the true signal much more closely.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, phi_diff, label="Differential phase")
plt.plot(t, signal, label="True signal", linewidth=2)
plt.xlabel("time")
plt.ylabel("phase")
plt.title("Differential context cancels common-mode noise")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "00_differential_recovery.png", dpi=200)
plt.show()

## 5. Quantify improvement

The individual sensor error is dominated by the common-mode noise. The differential error is set by the remaining independent local noise.

In [ ]:
individual_rmse = np.sqrt(np.mean((phi_a - 0.5 * signal) ** 2))
differential_rmse = np.sqrt(np.mean((phi_diff - signal) ** 2))

summary = pd.DataFrame({
    "quantity": ["individual_sensor_error", "differential_error"],
    "rmse": [individual_rmse, differential_rmse],
    "interpretation": [
        "dominated by common-mode noise",
        "common-mode noise cancels"
    ]
})

summary

In [ ]:
out_path = DATA / "00_context_summary.csv"
summary.to_csv(out_path, index=False)
out_path

## 6. README figure

This compact summary figure gives the repo an immediate visual foundation: differential readout reduces apparent error by rejecting shared noise.

In [ ]:
labels = ["Individual", "Differential"]
values = [individual_rmse, differential_rmse]

plt.figure(figsize=(6, 4))
plt.bar(labels, values)
plt.ylabel("RMSE")
plt.title("Differential readout reduces apparent error")
plt.tight_layout()
plt.savefig(FIGURES / "00_context_summary.png", dpi=200)
plt.show()

## 7. Takeaway

Shared noise is not automatically lost information.

If the measurement geometry identifies what is common and what is different, the common part can be rejected.

Notebook 07 extends this model by separating signal amplitude, common-mode noise amplitude, and local sensor noise.